# Blood–Brain Barrier Exclusion (BBBX) Score

Run the cells in order using Colab's normal CPU runtime. Setup is automatic; no session restart or Python-version selection is needed. Choose **SMILES**, **CSV**, **3D SDF**, or **Descriptor CSV** as input.
Training-set checks add flags only: compounds outside those criteria are still predicted.
Invalid inputs or failed structure preparation are reported in the output, and all input rows are retained.

SMILES inputs use the seeded structure preparation from notebook 01. Use an existing descriptor CSV
to replay previous numerical results; recalculating descriptors can introduce small differences.
The BBBX score is `(predicted_logBB + 2.5) / 4.5 × 6`, without clipping; it is not a probability.


In [ ]:
# @title Set Up Prediction Packages
# @markdown Run once, then continue through the notebook. No session restart or runtime-version selection is needed.
import json
import os
import subprocess
import sys
from pathlib import Path

BBBX_RUNTIME = Path("/content/bbbx_prediction_runtime")
BBBX_RUNTIME.mkdir(parents=True, exist_ok=True)
tools_dir = BBBX_RUNTIME / "tools"
uv = tools_dir / "bin/uv"
if not uv.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "--target", str(tools_dir), "uv==0.13.4"], check=True)

BBBX_PYTHON = BBBX_RUNTIME / "environment/bin/python"
if not BBBX_PYTHON.is_file():
    subprocess.run([str(uv), "venv", "--managed-python", "--python", "3.12.14",
                    str(BBBX_PYTHON.parent.parent)], check=True)
packages = ["numpy==1.26.4", "pandas==2.3.3", "scipy==1.15.2", "rdkit==2026.03.4",
            "networkx==2.8.8", "lightgbm==4.6.0", "joblib==1.5.3", "six==1.17.0",
            "setuptools==75.8.0"]
subprocess.run([str(uv), "pip", "install", "--python", str(BBBX_PYTHON), *packages], check=True)
subprocess.run([str(uv), "pip", "install", "--python", str(BBBX_PYTHON),
                "--no-deps", "mordred==1.2.0"], check=True)

# Check the packages in a fresh process. Colab's own NumPy is never replaced.
subprocess.run([str(BBBX_PYTHON), "-I", "-c",
    "import numpy, numpy.random, pandas, scipy, lightgbm, rdkit; from mordred import Calculator; "
    "assert numpy.__version__ == '1.26.4'; print('Prediction packages ready.')"], check=True)
print("Setup complete. Continue with Import Packages.")


In [ ]:
# @title Import Packages
import csv
import html
import io
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display

WORKER_PATH = BBBX_RUNTIME / "bbbx_worker.py"
WORKER_SOURCE = '# Import Packages\n\nimport base64\nimport hashlib\nimport io\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport zipfile\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport joblib\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom mordred import Calculator, descriptors\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem import Crippen, Descriptors, Lipinski, MACCSkeys, rdMolDescriptors\n\nRDLogger.DisableLog("rdApp.warning")\nBASE_SEED = 20260915\nDIELECTRIC_CONSTANT = 1\n\ndef training_set_flags(mol):\n    # Match notebook 01, including rounding before the numerical comparisons.\n    values = pd.Series({"MW": Descriptors.ExactMolWt(mol), "SLogP": Crippen.MolLogP(mol),\n                        "TopoPSA": rdMolDescriptors.CalcTPSA(mol)}).round(2)\n    checks = {\n        "fewer than 2 heavy atoms": mol.GetNumHeavyAtoms() >= 2,\n        "multiple disconnected fragments": len(Chem.GetMolFrags(mol)) == 1,\n        "no carbon atoms": any(a.GetAtomicNum() == 6 for a in mol.GetAtoms()),\n        "MW > 800": values["MW"] <= 800,\n        "H-bond donors > 6": Lipinski.NumHDonors(mol) <= 6,\n        "H-bond acceptors > 11": Lipinski.NumHAcceptors(mol) <= 11,\n        "SLogP outside (0, 5]": 0 < values["SLogP"] <= 5,\n        "TopoPSA >= 180": values["TopoPSA"] < 180,\n    }\n    return "; ".join(label for label, passed in checks.items() if not passed)\n\ndef connectivity(mol):\n    copy = Chem.Mol(mol)\n    Chem.RemoveStereochemistry(copy)\n    return Chem.MolToSmiles(Chem.RemoveHs(copy), isomericSmiles=False)\n\ndef find_java():\n    candidates = [Path(sys.prefix) / "bin/java"]\n    if os.environ.get("JAVA_HOME"):\n        candidates.append(Path(os.environ["JAVA_HOME"]) / "bin/java")\n    if shutil.which("java"):\n        candidates.append(Path(shutil.which("java")))\n    for candidate in candidates:\n        if candidate.is_file():\n            check = subprocess.run([str(candidate), "--list-modules"], capture_output=True, text=True)\n            if check.returncode == 0 and "jdk.compiler@" in check.stdout:\n                return str(candidate.resolve())\n    return None\n\n# Unchanged conformer helper from notebook 01.\nJAVA_SOURCE = r\'\'\'\nimport com.actelion.research.chem.MolfileV3Creator;\nimport com.actelion.research.chem.StereoMolecule;\nimport com.actelion.research.chem.SmilesParser;\nimport com.actelion.research.chem.forcefield.mmff.ForceFieldMMFF94;\nimport org.openmolecules.chem.conf.gen.ConformerGenerator;\nimport org.openmolecules.chem.conf.gen.RigidFragmentCache;\nimport java.io.*;\nimport java.nio.charset.StandardCharsets;\nimport java.nio.file.*;\nimport java.util.*;\n\npublic class SeededDataWarriorConformers {\n  static final int MAX_TORSION_SETS = 1000;\n  static final int MAX_ITERATIONS = 10000;\n  static final double GRADIENT_TOLERANCE = 0.0001;\n  static final double FUNCTION_TOLERANCE = 1.0e-6;\n\n  public static void main(String[] args) throws Exception {\n    Path manifest = Path.of(args[0]);\n    Path output = Path.of(args[1]);\n    Path failures = Path.of(args[2]);\n    double dielectric = Double.parseDouble(args[3]);\n    RigidFragmentCache.getDefaultInstance().loadDefaultCache();\n    ForceFieldMMFF94.initialize(ForceFieldMMFF94.MMFF94SPLUS);\n    Map<String,Object> options = new HashMap<>();\n    options.put("dielectric constant", dielectric);\n    int inputCount = 0, writtenCount = 0, failureCount = 0;\n\n    try (BufferedReader in = Files.newBufferedReader(manifest, StandardCharsets.UTF_8);\n         BufferedWriter sdf = Files.newBufferedWriter(output, StandardCharsets.UTF_8);\n         BufferedWriter failed = Files.newBufferedWriter(failures, StandardCharsets.UTF_8)) {\n      failed.write("dataframe_row\\tinput_id\\tsmiles\\terror\\n");\n      String line;\n      while ((line = in.readLine()) != null) {\n        inputCount++;\n        String[] f = line.split("\\\\t", -1);\n        String row = decode(f[0]), id = decode(f[1]), smiles = decode(f[2]);\n        long seed = Long.parseLong(f[3]);\n        String originalProperties = decode(f[4]);\n        try {\n          StereoMolecule mol = new StereoMolecule();\n          new SmilesParser().parse(mol, smiles);\n          ConformerGenerator cg = new ConformerGenerator(seed, true);\n          if (!cg.initializeConformers(mol, ConformerGenerator.STRATEGY_LIKELY_SYSTEMATIC,\n                  MAX_TORSION_SETS, false))\n            throw new Exception("Conformer generator could not initialize");\n          mol = cg.getNextConformerAsMolecule(mol);\n          if (mol == null) throw new Exception("No conformer generated");\n          double energy = Double.NaN;\n          String minimizationError = "";\n\n          try {\n            energy = minimize(mol, options);\n          } catch (Exception e) {\n            // Match DataWarrior: retain the generated conformer when\n            // MMFF94 typing or minimization fails.\n            minimizationError = e.getClass().getSimpleName() + ": " + e.getMessage();\n          }\n\n          center(mol);\n          mol.setName(id.isBlank() ? "row " + row : id);\n          sdf.write(new MolfileV3Creator(mol).getMolfile());\n          // Restore every original CSV column using its exact original name.\n          for (String entry : originalProperties.split("\\u001e", -1)) {\n            if (entry.isEmpty()) continue;\n\n            int separator = entry.indexOf(\'\\u001f\');\n            if (separator < 0) continue;\n\n            String columnName = entry.substring(0, separator);\n            String columnValue = entry.substring(separator + 1);\n\n            property(sdf, columnName, columnValue);\n          }\n          property(sdf, "DATAFRAME_ROW", row);\n          property(sdf, "INPUT_ID", id);\n          property(sdf, "INPUT_SMILES", smiles);\n          property(sdf, "CONFORMER_SEED", Long.toString(seed));\n          property(sdf, "CONFORMER_ALGORITHM", "systematic, low-energy biased");\n          property(sdf, "TORSION_SOURCE", "crystallographic");\n          property(sdf, "FORCE_FIELD", "MMFF94s+");\n          property(sdf, "DIELECTRIC_CONSTANT", Double.toString(dielectric));\n          property(\n              sdf,\n              "MMFF94_ENERGY",\n              Double.isNaN(energy) ? "" : String.format(Locale.ROOT, "%.9g", energy)\n          );\n          property(sdf, "MINIMIZATION_ERROR", minimizationError);\n          sdf.write("$$$$\\n");\n          writtenCount++;\n        } catch (Throwable e) {\n          failureCount++;\n          failed.write(clean(row)+"\\t"+clean(id)+"\\t"+clean(smiles)+"\\t"\n              +clean(e.getClass().getSimpleName()+": "+e.getMessage())+"\\n");\n        }\n      }\n    }\n    System.out.printf(Locale.ROOT, "Input: %d  Written: %d  Failed: %d%n",\n        inputCount, writtenCount, failureCount);\n    if (failureCount != 0) System.exit(1);\n  }\n\n  static double minimize(StereoMolecule mol, Map<String,Object> options) throws Exception {\n    int[] fragmentNo = new int[mol.getAllAtoms()];\n    int fragmentCount = mol.getFragmentNumbers(fragmentNo, false, true);\n    double energy = Double.NaN;\n    if (fragmentCount == 1) {\n      ForceFieldMMFF94 ff = new ForceFieldMMFF94(mol, ForceFieldMMFF94.MMFF94SPLUS, options);\n      int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);\n      if (error != 0) throw new Exception("MMFF94 error code " + error);\n      energy = ff.getTotalEnergy();\n    } else {\n      StereoMolecule[] fragments = mol.getFragments(fragmentNo, fragmentCount);\n      int maxAtoms = 0;\n      for (StereoMolecule fragment : fragments) {\n        if (fragment.getAllAtoms() > 2) {\n          ForceFieldMMFF94 ff = new ForceFieldMMFF94(fragment, ForceFieldMMFF94.MMFF94SPLUS, options);\n          int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);\n          if (error != 0) throw new Exception("MMFF94 error code " + error);\n          if (maxAtoms < fragment.getAllAtoms()) {\n            maxAtoms = fragment.getAllAtoms(); energy = ff.getTotalEnergy();\n          }\n        }\n      }\n      int[] atomInFragment = new int[fragmentCount];\n      for (int atom=0; atom<fragmentNo.length; atom++) {\n        int fragment = fragmentNo[atom], sourceAtom = atomInFragment[fragment]++;\n        mol.setAtomX(atom, fragments[fragment].getAtomX(sourceAtom));\n        mol.setAtomY(atom, fragments[fragment].getAtomY(sourceAtom));\n        mol.setAtomZ(atom, fragments[fragment].getAtomZ(sourceAtom));\n      }\n    }\n    return energy;\n  }\n\n  static void center(StereoMolecule mol) {\n    double x=0, y=0, z=0;\n    for (int a=0; a<mol.getAllAtoms(); a++) { x+=mol.getAtomX(a); y+=mol.getAtomY(a); z+=mol.getAtomZ(a); }\n    x/=mol.getAllAtoms(); y/=mol.getAllAtoms(); z/=mol.getAllAtoms();\n    for (int a=0; a<mol.getAllAtoms(); a++) {\n      mol.setAtomX(a,mol.getAtomX(a)-x); mol.setAtomY(a,mol.getAtomY(a)-y); mol.setAtomZ(a,mol.getAtomZ(a)-z);\n    }\n  }\n\n  static void property(BufferedWriter w, String name, String value) throws Exception {\n    w.write(">  <"+name+">\\n"+clean(value)+"\\n\\n");\n  }\n  static String decode(String s) { return new String(Base64.getDecoder().decode(s), StandardCharsets.UTF_8); }\n  static String clean(String s) { return s == null ? "" : s.replace(\'\\t\',\' \').replace(\'\\r\',\' \').replace(\'\\n\',\' \'); }\n}\n\'\'\'\n\nimport pickle\nimport json\n\n# Retain SDF properties and full coordinate precision between notebook steps.\nChem.SetDefaultPickleProperties(Chem.PropertyPickleOptions.AllProps | Chem.PropertyPickleOptions.CoordsAsDouble)\nSTAGES = {\'read\': \'rows, supplied_molecules = [], {}\\nif input_kind == "SMILES":\\n    smiles = SMILES_space_separated.split()\\n    names = [s.strip() for s in mol_names_optional.split(",")] if mol_names_optional.strip() else []\\n    if names and len(names) != len(smiles):\\n        raise ValueError("Provide one comma-separated name per SMILES, or leave names blank.")\\n    rows = [{"name": names[i] if names else f"compound_{i+1}", "smiles": smi, "error": ""}\\n            for i, smi in enumerate(smiles)]\\nelif input_kind in {"CSV", "Descriptor CSV"}:\\n    table = pd.read_csv(input_file, dtype=str, keep_default_na=False)\\n    if smiles_column not in table:\\n        raise ValueError(f"Missing SMILES column: {smiles_column}")\\n    rows = [{"name": row.get(name_column, "") or f"compound_{i+1}",\\n             "smiles": row[smiles_column].strip(), "error": ""} for i, row in table.iterrows()]\\nelif input_kind == "3D SDF":\\n    # Preserve explicit hydrogens, as in the original Mordred CLI workflow.\\n    for i, mol in enumerate(Chem.SDMolSupplier(str(input_file), removeHs=False)):\\n        row = {"name": f"compound_{i+1}", "smiles": "", "error": ""}\\n        if mol is None:\\n            row["error"] = "Invalid SDF record."\\n        else:\\n            row["name"] = mol.GetProp("_Name") if mol.HasProp("_Name") else row["name"]\\n            row["smiles"] = next((mol.GetProp(key).strip() for key in ("INPUT_SMILES", "SMILES")\\n                                   if mol.HasProp(key) and mol.GetProp(key).strip()), Chem.MolToSmiles(Chem.RemoveHs(mol)))\\n            parsed = Chem.MolFromSmiles(row["smiles"])\\n            if parsed is None or connectivity(parsed) != connectivity(mol):\\n                row["error"] = "SDF structure and its SMILES property do not match."\\n            elif not mol.GetNumConformers() or not mol.GetConformer().Is3D():\\n                row["error"] = "A 3D SDF is required; use SMILES/CSV to generate coordinates."\\n            elif not np.isfinite(mol.GetConformer().GetPositions()).all():\\n                row["error"] = "Nonfinite SDF coordinates."\\n            supplied_molecules[i+1] = mol\\n        rows.append(row)\\nelse:\\n    raise ValueError("Choose SMILES, CSV, 3D SDF, or Descriptor CSV.")\\nif not rows:\\n    raise ValueError("No input compounds were provided.")\\ninput_df = pd.DataFrame(rows)\\ninput_df.index = pd.Index(range(1, len(input_df)+1), name="input_row")\\nprint(f"Read {len(input_df)} compounds.")\\n\', \'prepare\': \'# @title Prepare Molecules\\n\\ntimestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")\\ninput_df["status"] = "ready"\\ninput_df["training_set_flags"] = ""\\ninput_df["matches_training_criteria"] = pd.Series(pd.NA, index=input_df.index, dtype="boolean")\\nfor row_id, row in input_df.iterrows():\\n    mol = Chem.MolFromSmiles(row["smiles"]) if row["smiles"] else None\\n    if row["error"] or mol is None or mol.GetNumAtoms() == 0:\\n        input_df.loc[row_id, "status"] = "invalid_input"\\n        input_df.loc[row_id, "error"] = row["error"] or "Invalid or empty SMILES."\\n    else:\\n        flags = training_set_flags(mol)\\n        input_df.loc[row_id, "training_set_flags"] = flags\\n        input_df.loc[row_id, "matches_training_criteria"] = not bool(flags)\\n\\n# Training-set flags do NOT remove compounds from the prediction list.\\nvalid_ids = input_df.index[input_df["status"].eq("ready")].tolist()\\nmolecules = {i: supplied_molecules[i] for i in valid_ids} if input_kind == "3D SDF" else {}\\nif input_kind in {"SMILES", "CSV"} and valid_ids:\\n    java = find_java()\\n    if java is None:\\n        subprocess.run(["apt-get", "update", "-qq"], check=True)\\n        subprocess.run(["apt-get", "install", "-qq", "-y", "openjdk-21-jdk-headless"], check=True)\\n        java = find_java()\\n    if java is None:\\n        raise RuntimeError("A Java JDK is required to generate 3D structures.")\\n    jar = Path(OPEN_CHEM_LIB_JAR).expanduser().resolve()\\n    if not jar.is_file():\\n        raise FileNotFoundError("Set OPEN_CHEM_LIB_JAR to the DataWarrior JAR used in the dataset workflow.")\\n    sdf_path = OUTPUT_DIR / f"prepared_3d_{timestamp}.sdf"\\n    failure_path = OUTPUT_DIR / f"conformer_failures_{timestamp}.tsv"\\n    encode = lambda value: base64.b64encode(str(value).encode()).decode()\\n    with tempfile.TemporaryDirectory(prefix="bbbx_3d_") as temporary:\\n        temporary = Path(temporary)\\n        source = temporary / "SeededDataWarriorConformers.java"\\n        source.write_text(JAVA_SOURCE, encoding="utf-8")\\n        manifest = temporary / "input.tsv"\\n        with manifest.open("w", encoding="utf-8") as handle:\\n            for row_id in valid_ids:\\n                row = input_df.loc[row_id]\\n                seed = int.from_bytes(hashlib.sha256(f"{BASE_SEED}\\\\0{row[\\\'smiles\\\']}".encode()).digest()[:8], "big", signed=True) or 1\\n                properties = f"INPUT_NAME\\\\x1f{row[\\\'name\\\']}\\\\x1eSMILES\\\\x1f{row[\\\'smiles\\\']}"\\n                handle.write("\\\\t".join([encode(row_id), encode(row_id), encode(row["smiles"]), str(seed), encode(properties)]) + "\\\\n")\\n        result = subprocess.run([java, "--class-path", str(jar), str(source), str(manifest),\\n                                 str(sdf_path), str(failure_path), str(DIELECTRIC_CONSTANT)], capture_output=True, text=True)\\n    if result.returncode not in (0, 1) or not failure_path.is_file():\\n        raise RuntimeError(result.stderr or "Conformer generation could not run.")\\n    failures = pd.read_csv(failure_path, sep="\\\\t", dtype=str, keep_default_na=False)\\n    for _, failure in failures.iterrows():\\n        row_id = int(failure["dataframe_row"])\\n        input_df.loc[row_id, ["status", "error"]] = ["preparation_failed", failure["error"]]\\n    if sdf_path.is_file() and sdf_path.stat().st_size:\\n        for mol in Chem.SDMolSupplier(str(sdf_path), removeHs=False):\\n            if mol is None or not mol.HasProp("DATAFRAME_ROW"):\\n                raise RuntimeError("A generated SDF record lost its input-row identifier.")\\n            row_id = int(mol.GetProp("DATAFRAME_ROW"))\\n            if row_id not in valid_ids or row_id in molecules or input_df.loc[row_id, "status"] != "ready":\\n                raise RuntimeError("Generated structures do not align with input rows.")\\n            if mol.GetProp("INPUT_SMILES") != input_df.loc[row_id, "smiles"]:\\n                raise RuntimeError("Generated SMILES do not align with input rows.")\\n            molecules[row_id] = mol\\n    if set(molecules) | set(failures["dataframe_row"].astype(int)) != set(valid_ids):\\n        raise RuntimeError("Some conformer outcomes are unaccounted for.")\\n    print(result.stdout.strip())\\n\\nflagged = input_df["training_set_flags"].ne("").sum()\\nprint(f"{flagged} compound(s) flagged against the training criteria; prediction will still be attempted.")\\n\', \'descriptors\': \'# Calculate Descriptors\\n\\n# Load each fold\\\'s saved feature order; do not select columns from the input batch.\\nmodels = []\\nfor model_id in range(5):\\n    saved = joblib.load(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}_preprocessor.joblib")\\n    names = list(saved["feature_names"])\\n    if saved.get("missing_value_policy") != "native_lightgbm_nan_handling_no_imputation":\\n        raise ValueError(f"Fold {model_id}: use the updated model/preprocessor pair.")\\n    if float(saved.get("threshold", -1)) != -1 or len(names) != len(set(names)):\\n        raise ValueError(f"Fold {model_id}: unexpected model metadata.")\\n    model = lgb.Booster(model_file=str(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}.txt"))\\n    if model.num_feature() != len(names):\\n        raise ValueError(f"Fold {model_id}: model and feature list do not match.")\\n    models.append((model, names))\\nfeature_names = list(dict.fromkeys(name for _, names in models for name in names))\\n\\nif input_kind == "Descriptor CSV":\\n    # Preserve the default pandas CSV parser used by prediction script 11.\\n    supplied = pd.read_csv(input_file, low_memory=False)\\n    if len(supplied) != len(input_df):\\n        raise ValueError("Input and descriptor row counts differ.")\\n    missing = [name for name in feature_names if name not in supplied]\\n    if missing:\\n        raise ValueError(f"Missing {len(missing)} required descriptors; first missing: {missing[:10]}")\\n    supplied.index = input_df.index\\n    fully_labelled_df = supplied.loc[valid_ids, feature_names].apply(pd.to_numeric, errors="coerce")\\nelse:\\n    registry = {str(d): d for d in Calculator(descriptors, ignore_3D=False).descriptors}\\n    mordred_names = [name for name in feature_names if name in registry]\\n    fp_names = {f"ECFP_{i}" for i in range(2048)} | {f"MACCS_{i}" for i in range(167)}\\n    if set(feature_names) - set(mordred_names) - fp_names:\\n        raise ValueError("The saved feature list contains an unknown descriptor.")\\n    calculator = Calculator([registry[name] for name in mordred_names], ignore_3D=False)\\n    feature_rows = []\\n    for row_id, mol in molecules.items():\\n        two_d = Chem.MolFromSmiles(input_df.loc[row_id, "smiles"])\\n        ecfp = rdMolDescriptors.GetMorganFingerprintAsBitVect(two_d, 2, nBits=2048, useChirality=False, useFeatures=False)\\n        maccs = MACCSkeys.GenMACCSKeys(two_d)\\n        values = {f"ECFP_{i}": int(ecfp[i]) for i in range(2048)}\\n        values.update({f"MACCS_{i}": int(maccs[i]) for i in range(167)})\\n        for name, value in zip(mordred_names, calculator(mol)):\\n            try:\\n                number = float(value)\\n            except (ValueError, TypeError):\\n                number = np.nan\\n            values[name] = (str(int(value)) if isinstance(value, (bool, np.bool_)) else str(value)) if np.isfinite(number) else ""\\n        feature_rows.append(pd.Series(values, name=row_id))\\n    fully_labelled_df = pd.DataFrame(feature_rows, columns=feature_names)\\n    fully_labelled_df.index.name = "input_row"\\n    # Match the Mordred CSV read, then assembled-dataset CSV read in the original workflow.\\n    if len(fully_labelled_df):\\n        for _ in range(2):\\n            fully_labelled_df = pd.read_csv(io.StringIO(fully_labelled_df.to_csv()), index_col="input_row", low_memory=False)\\n    fully_labelled_df = fully_labelled_df.apply(pd.to_numeric, errors="coerce")\\n\\nfully_labelled_df = fully_labelled_df.replace([np.inf, -np.inf], np.nan)\\nfully_labelled_df.index.name = "input_row"\\ndescriptor_file = OUTPUT_DIR / f"fully_labelled_dataset_{timestamp}.csv"\\nmetadata = input_df.loc[fully_labelled_df.index, ["name", "smiles"]].rename(columns={"name": "Name", "smiles": "SMILES"})\\nmetadata.join(fully_labelled_df).to_csv(descriptor_file)\\nprint(f"Descriptors saved to: {descriptor_file}")\\n\', \'predict\': \'# Get Score Values\\n\\noutput_df = input_df.copy()\\noutput_df["predicted_logBB"] = np.nan\\noutput_df["BBBX_score"] = np.nan\\noutput_df["predicted_BBB_label"] = ""\\nall_predictions = []\\nif len(fully_labelled_df):\\n    for model, names in models:\\n        # Missing descriptor values stay NaN; LightGBM handles them natively.\\n        X_all = fully_labelled_df.loc[:, names].to_numpy(dtype=float)\\n        all_predictions.append(model.predict(X_all, num_threads=1))\\n    all_preds = np.mean(np.asarray(all_predictions), axis=0)\\n    if not np.isfinite(all_preds).all():\\n        raise RuntimeError("The model returned a nonfinite prediction.")\\n    predicted_ids = fully_labelled_df.index\\n    output_df.loc[predicted_ids, "predicted_logBB"] = all_preds\\n    output_df.loc[predicted_ids, "BBBX_score"] = (all_preds + 2.5) / 4.5 * 6\\n    output_df.loc[predicted_ids, "predicted_BBB_label"] = np.where(all_preds < -1, "nonpenetrant", "penetrant")\\n    output_df.loc[predicted_ids, "status"] = "predicted"\\n\\noutput_df = output_df.reset_index()[["input_row", "name", "smiles", "predicted_logBB", "BBBX_score",\\n    "predicted_BBB_label", "matches_training_criteria", "training_set_flags", "status", "error"]]\\nunique_file_name = OUTPUT_DIR / f"predicted_scores_{timestamp}.csv"\\noutput_df.to_csv(unique_file_name, index=False)\\nprint(f"Predictions saved to: {unique_file_name}")\\n\'}\nMODEL_LOADING = \'# Calculate Descriptors\\n\\n# Load each fold\\\'s saved feature order; do not select columns from the input batch.\\nmodels = []\\nfor model_id in range(5):\\n    saved = joblib.load(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}_preprocessor.joblib")\\n    names = list(saved["feature_names"])\\n    if saved.get("missing_value_policy") != "native_lightgbm_nan_handling_no_imputation":\\n        raise ValueError(f"Fold {model_id}: use the updated model/preprocessor pair.")\\n    if float(saved.get("threshold", -1)) != -1 or len(names) != len(set(names)):\\n        raise ValueError(f"Fold {model_id}: unexpected model metadata.")\\n    model = lgb.Booster(model_file=str(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}.txt"))\\n    if model.num_feature() != len(names):\\n        raise ValueError(f"Fold {model_id}: model and feature list do not match.")\\n    models.append((model, names))\\nfeature_names = list(dict.fromkeys(name for _, names in models for name in names))\\n\\n\'\n\n\ndef main():\n    stage, config_path = sys.argv[1:]\n    config_path = Path(config_path)\n    config = json.loads(config_path.read_text())\n    state_path = config_path.with_suffix(".pkl")\n    result_path = config_path.with_suffix(".result.json")\n    state = {} if stage == "read" else pickle.loads(state_path.read_bytes())\n    order = {"read": 0, "prepare": 1, "descriptors": 2, "predict": 3}\n    if stage != "read" and order.get(state.get("completed_stage"), -1) < order[stage] - 1:\n        raise RuntimeError("Run the preceding notebook cell first.")\n    ns = dict(globals())\n    ns.update(config)\n    ns.update(state)\n    for name in ("MODEL_DIR", "OPEN_CHEM_LIB_JAR", "OUTPUT_DIR"):\n        ns[name] = Path(config[name])\n    ns["input_file"] = Path(config["input_file"]) if config["input_file"] else None\n    if stage == "prepare":\n        ns["input_df"] = state["raw_input_df"].copy(deep=True)\n    if stage == "predict":\n        exec(MODEL_LOADING, ns)\n    exec(STAGES[stage], ns)\n    keep = {"input_df", "raw_input_df", "supplied_molecules", "valid_ids", "molecules", "timestamp",\n            "fully_labelled_df", "descriptor_file", "unique_file_name"}\n    if stage == "read":\n        ns["raw_input_df"] = ns["input_df"].copy(deep=True)\n    state = {name: ns[name] for name in keep if name in ns}\n    if stage in {"read", "prepare"}:\n        for name in ("fully_labelled_df", "descriptor_file", "unique_file_name"):\n            state.pop(name, None)\n    elif stage == "descriptors":\n        state.pop("unique_file_name", None)\n    state["completed_stage"] = stage\n    temporary = state_path.with_suffix(".tmp")\n    temporary.write_bytes(pickle.dumps(state, protocol=5))\n    temporary.replace(state_path)\n    result = {"stage": stage, "input_count": len(ns["input_df"])}\n    if stage == "predict":\n        result["predictions_csv"] = str(ns["unique_file_name"])\n    result_path.write_text(json.dumps(result))\n\nif __name__ == "__main__":\n    main()\n'
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")

def run_step(stage):
    result_path = CONFIG_PATH.with_suffix(".result.json")
    result_path.unlink(missing_ok=True)
    result = subprocess.run([str(BBBX_PYTHON), "-I", str(WORKER_PATH), stage, str(CONFIG_PATH)],
                             capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or f"The {stage} step failed.")
    return json.loads(result_path.read_text())

print("Prediction workflow loaded.")


In [ ]:
# @title Clone Repository
from google.colab import files

PROJECT_ROOT = Path("/content/bbb-exc-score")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/saraziadatt/bbb-exc-score.git", str(PROJECT_ROOT)], check=True)
MODEL_DIR = PROJECT_ROOT / "final_model/models"
OPEN_CHEM_LIB_JAR = PROJECT_ROOT / "tools/datawarrior_all.jar"
OUTPUT_DIR = PROJECT_ROOT / "final_model/predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This upload is needed only until the updated models/JAR are in the repository.
required = {f"custom_mtl_bab_outer_{fold}{suffix}": MODEL_DIR / f"custom_mtl_bab_outer_{fold}{suffix}"
            for fold in range(5) for suffix in (".txt", "_preprocessor.joblib")}
required["datawarrior_all.jar"] = OPEN_CHEM_LIB_JAR
if not all(path.is_file() for path in required.values()):
    print("Upload BBBX_Colab_update.zip once to supply the updated model files.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one model bundle ZIP.")
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as bundle:
        for name, destination in required.items():
            matches = [entry for entry in bundle.infolist() if Path(entry.filename).name == name
                       and not entry.filename.startswith("__MACOSX/") and not entry.is_dir()]
            if len(matches) != 1:
                raise ValueError(f"The bundle must contain exactly one {name}.")
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(matches[0]) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
    del uploaded
print("Model files ready.")


In [ ]:
# @title Read Input
# @markdown Enter SMILES separated by spaces and optional names separated by commas. For CSV/SDF, upload one file.

input_kind = "SMILES" # @param ["SMILES", "CSV", "3D SDF", "Descriptor CSV"]
SMILES_space_separated = "CC(=O)Nc1ccc(O)cc1" # @param {type:"string"}
mol_names_optional = "" # @param {type:"string"}
smiles_column = "SMILES" # @param {type:"string"}
name_column = "Name" # @param {type:"string"}
input_file = None
if input_kind != "SMILES":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one input file.")
    filename, contents = next(iter(uploaded.items()))
    input_file = OUTPUT_DIR / ("input_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + "_" + Path(filename).name)
    input_file.write_bytes(contents)
    del uploaded, contents


# Keep inputs and intermediate calculations in the separate prediction environment.
CONFIG_PATH = BBBX_RUNTIME / ("input_" + uuid4().hex + ".json")
CONFIG_PATH.write_text(json.dumps({
    "input_kind": input_kind, "SMILES_space_separated": SMILES_space_separated,
    "mol_names_optional": mol_names_optional, "smiles_column": smiles_column, "name_column": name_column,
    "input_file": str(input_file.resolve()) if input_file else None,
    "MODEL_DIR": str(MODEL_DIR.resolve()), "OPEN_CHEM_LIB_JAR": str(OPEN_CHEM_LIB_JAR.resolve()),
    "OUTPUT_DIR": str(OUTPUT_DIR.resolve()),
}))
globals().pop("unique_file_name", None)
_ = run_step("read")


In [ ]:
# @title Prepare Molecules
globals().pop("unique_file_name", None)
_ = run_step("prepare")


In [ ]:
# @title Calculate Descriptors
globals().pop("unique_file_name", None)
_ = run_step("descriptors")


In [ ]:
# @title Get Score Values
result = run_step("predict")
unique_file_name = Path(result["predictions_csv"])


In [ ]:
# @title Display Output
with unique_file_name.open(newline="", encoding="utf-8") as handle:
    output_rows = list(csv.DictReader(handle))
columns = list(output_rows[0]) if output_rows else []
header = "".join("<th>" + html.escape(name) + "</th>" for name in columns)
body = "".join("<tr>" + "".join("<td>" + html.escape(row[name]) + "</td>" for name in columns) + "</tr>"
               for row in output_rows[:200])
display(HTML('<div style="overflow:auto;max-height:600px"><table class="dataframe"><thead><tr>'
             + header + "</tr></thead><tbody>" + body + "</tbody></table></div>"))
if len(output_rows) > 200:
    print(f"Showing 200 of {len(output_rows)} rows. The downloaded CSV contains every row.")


In [ ]:
# @title Download Predictions
files.download(str(unique_file_name))
